# Modeling

This notebook trains and evaluates baseline and machine learning models for predicting season long full-PPR fantasy points.

Models are evaluated using time based validation so that each validation season is predicted only from earlier seasons.

In [117]:
import polars as pl

df = pl.read_csv("../data/processed/modeling_dataset_2018_2025.csv")

print(df.shape)
df.group_by("season").len().sort("season")

(2909, 83)


season,len
i64,u32
2018,307
2019,317
2020,362
2021,389
2022,394
2023,374
2024,368
2025,398


## Baseline

Baseline model just using last season's fantasy points to predict this season

In [118]:
baseline_df = df.filter(
    pl.col("fantasy_points_lag_1").is_not_null()
).with_columns(
    pl.col("fantasy_points_lag_1")
      .alias("baseline_prediction")
)

In [119]:
baseline_metrics = baseline_df.select([
    (
        pl.col("fantasy_points_ppr_calc")
        - pl.col("baseline_prediction")
    )
    .abs()
    .mean()
    .alias("mae"),

    (
        (
            pl.col("fantasy_points_ppr_calc")
            - pl.col("baseline_prediction")
        ) ** 2
    )
    .mean()
    .sqrt()
    .alias("rmse")
])

baseline_metrics

mae,rmse
f64,f64
47.957016,68.482623


In [120]:
baseline_by_position = (
    baseline_df
    .group_by("position")
    .agg([
        (
            pl.col("fantasy_points_ppr_calc")
            - pl.col("baseline_prediction")
        )
        .abs()
        .mean()
        .alias("mae"),

        (
            (
                pl.col("fantasy_points_ppr_calc")
                - pl.col("baseline_prediction")
            ) ** 2
        )
        .mean()
        .sqrt()
        .alias("rmse")
    ])
    .sort("position")
)

baseline_by_position

position,mae,rmse
str,f64,f64
"""QB""",64.99386,92.610864
"""RB""",53.239807,74.687359
"""TE""",32.863969,45.634756
"""WR""",46.546404,64.023121


70 / 30 Weighted Baseline from EDA

In [121]:
weighted_baseline_df = df.filter(
    pl.col("fantasy_points_2yr_weighted").is_not_null()
).with_columns(
    pl.col("fantasy_points_2yr_weighted")
      .alias("baseline_prediction")
)

In [122]:
weighted_baseline_metrics = weighted_baseline_df.select([
    (
        pl.col("fantasy_points_ppr_calc")
        - pl.col("baseline_prediction")
    )
    .abs()
    .mean()
    .alias("mae"),

    (
        (
            pl.col("fantasy_points_ppr_calc")
            - pl.col("baseline_prediction")
        ) ** 2
    )
    .mean()
    .sqrt()
    .alias("rmse")
])

weighted_baseline_metrics

mae,rmse
f64,f64
46.043621,64.551699


In [123]:
weighted_baseline_by_position = (
    weighted_baseline_df
    .group_by("position")
    .agg([
        (
            pl.col("fantasy_points_ppr_calc")
            - pl.col("baseline_prediction")
        )
        .abs()
        .mean()
        .alias("mae"),

        (
            (
                pl.col("fantasy_points_ppr_calc")
                - pl.col("baseline_prediction")
            ) ** 2
        )
        .mean()
        .sqrt()
        .alias("rmse")
    ])
    .sort("position")
)

weighted_baseline_by_position

position,mae,rmse
str,f64,f64
"""QB""",60.485065,84.773374
"""RB""",51.863113,71.442984
"""TE""",31.186835,43.249383
"""WR""",45.14853,60.838066


In [124]:
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [125]:
model_data = df.to_pandas()

# RB only model to start

In [126]:
rb_features = [
    "age",
    "age_squared",
    "years_exp",
    "draft_number_filled",
    "undrafted_flag",
    "team_change_flag",

    "fantasy_points_lag_1",
    "fantasy_points_lag_2",
    "fantasy_ppg_lag_1",
    "fantasy_points_2yr_weighted",

    "games_lag_1",
    "games_2yr_avg",

    "fantasy_points_change",

    "targets_lag_1",
    "carries_lag_1",
    "receptions_lag_1",
    "opportunities_lag_1",

    "targets_per_game_lag_1",
    "carries_per_game_lag_1",
    "opportunities_per_game_lag_1",

    "yards_per_carry_lag_1",
    "fantasy_points_per_opportunity_lag_1",

    "targets_2yr_avg",
    "carries_2yr_avg",
    "opportunities_2yr_avg"
]

target = "fantasy_points_ppr_calc"

In [127]:
rb_df = model_data[
    model_data["position"] == "RB"
].copy()

In [128]:
validation_seasons = [2021, 2022, 2023, 2024, 2025]

In [129]:
rb_linear_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LinearRegression())
])

In [130]:
rb_results = []

for validation_season in validation_seasons:

    train = rb_df[
        rb_df["season"] < validation_season
    ]

    valid = rb_df[
        rb_df["season"] == validation_season
    ]

    X_train = train[rb_features]
    y_train = train[target]

    X_valid = valid[rb_features]
    y_valid = valid[target]

    rb_linear_model.fit(X_train, y_train)

    predictions = rb_linear_model.predict(X_valid)

    mae = mean_absolute_error(
        y_valid,
        predictions
    )

    rmse = mean_squared_error(
        y_valid,
        predictions
    ) ** 0.5

    rb_results.append({
        "season": validation_season,
        "train_rows": len(train),
        "validation_rows": len(valid),
        "mae": mae,
        "rmse": rmse
    })

In [131]:
rb_results_df = pd.DataFrame(rb_results)

rb_results_df

,season,train_rows,validation_rows,mae,rmse
0,2021,250,99,44.517064,58.599303
1,2022,349,100,48.096662,65.477522
2,2023,449,93,45.666090,59.796570
3,2024,542,89,49.025673,67.242573
4,2025,631,95,48.735904,66.178768


In [132]:
rb_results_df[
    ["mae", "rmse"]
].mean()

mae     47.208279
rmse    63.458947
dtype: float64

In [133]:
rb_baseline_results = []

for validation_season in validation_seasons:

    valid = rb_df[
        rb_df["season"] == validation_season
    ].copy()

    y_valid = valid[target]

    baseline_predictions = valid[
        "fantasy_points_2yr_weighted"
    ]

    mae = mean_absolute_error(
        y_valid,
        baseline_predictions
    )

    rmse = mean_squared_error(
        y_valid,
        baseline_predictions
    ) ** 0.5

    rb_baseline_results.append({
        "season": validation_season,
        "baseline_mae": mae,
        "baseline_rmse": rmse
    })

rb_baseline_results_df = pd.DataFrame(
    rb_baseline_results
)

rb_baseline_results_df

,season,baseline_mae,baseline_rmse
0,2021,52.160929,67.150246
1,2022,49.155720,68.551232
2,2023,49.609785,71.158244
3,2024,46.841056,68.120926
4,2025,48.621789,68.609732


In [134]:
rb_comparison = rb_results_df.merge(
    rb_baseline_results_df,
    on="season"
)

rb_comparison

,season,train_rows,validation_rows,mae,rmse,baseline_mae,baseline_rmse
0,2021,250,99,44.517064,58.599303,52.160929,67.150246
1,2022,349,100,48.096662,65.477522,49.155720,68.551232
2,2023,449,93,45.666090,59.796570,49.609785,71.158244
3,2024,542,89,49.025673,67.242573,46.841056,68.120926
4,2025,631,95,48.735904,66.178768,48.621789,68.609732


In [135]:
rb_comparison["mae_improvement"] = (
    rb_comparison["baseline_mae"]
    - rb_comparison["mae"]
)

rb_comparison["rmse_improvement"] = (
    rb_comparison["baseline_rmse"]
    - rb_comparison["rmse"]
)

rb_comparison

,season,train_rows,validation_rows,mae,rmse,baseline_mae,baseline_rmse,mae_improvement,rmse_improvement
0,2021,250,99,44.517064,58.599303,52.160929,67.150246,7.643866,8.550943
1,2022,349,100,48.096662,65.477522,49.155720,68.551232,1.059058,3.073710
2,2023,449,93,45.666090,59.796570,49.609785,71.158244,3.943695,11.361674
3,2024,542,89,49.025673,67.242573,46.841056,68.120926,-2.184617,0.878354
4,2025,631,95,48.735904,66.178768,48.621789,68.609732,-0.114115,2.430963


In [136]:
rb_comparison[[
    "mae",
    "baseline_mae",
    "mae_improvement",
    "rmse",
    "baseline_rmse",
    "rmse_improvement"
]].mean()

mae                 47.208279
baseline_mae        49.277856
mae_improvement      2.069577
rmse                63.458947
baseline_rmse       68.718076
rmse_improvement     5.259129
dtype: float64

# Refactor RB walk-forward validation into a reusable function

In [137]:
def run_walk_forward_linear_model(
    data,
    position,
    features,
    target,
    validation_seasons
):
    """
    Evaluate a position-specific linear regression model using
    walk-forward validation.

    For each validation season, the model trains only on earlier seasons.
    """

    position_data = data[
        data["position"] == position
    ].copy()

    fold_results = []

    for validation_season in validation_seasons:

        train = position_data[
            position_data["season"] < validation_season
        ].copy()

        valid = position_data[
            position_data["season"] == validation_season
        ].copy()

        assert train["season"].max() < validation_season
        assert valid["season"].nunique() == 1
        assert valid["season"].iloc[0] == validation_season

        if train.empty or valid.empty:
            continue

        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LinearRegression())
        ])

        model.fit(
            train[features],
            train[target]
        )

        predictions = model.predict(valid[features])

        fold_results.append({
            "position": position,
            "validation_season": validation_season,
            "train_rows": len(train),
            "validation_rows": len(valid),
            "mae": mean_absolute_error(
                valid[target],
                predictions
            ),
            "rmse": mean_squared_error(
                valid[target],
                predictions
            ) ** 0.5
        })

    fold_results = pd.DataFrame(fold_results)

    summary = pd.DataFrame([{
        "position": position,
        "mean_mae": fold_results["mae"].mean(),
        "mean_rmse": fold_results["rmse"].mean(),
        "validation_folds": len(fold_results)
    }])

    return fold_results, summary

In [138]:
rb_fold_results, rb_summary = run_walk_forward_linear_model(
    data=model_data,
    position="RB",
    features=rb_features,
    target=target,
    validation_seasons=validation_seasons
)

display(rb_fold_results)
display(rb_summary)

,position,validation_season,train_rows,validation_rows,mae,rmse
0,RB,2021,250,99,44.517064,58.599303
1,RB,2022,349,100,48.096662,65.477522
2,RB,2023,449,93,45.666090,59.796570
3,RB,2024,542,89,49.025673,67.242573
4,RB,2025,631,95,48.735904,66.178768


,position,mean_mae,mean_rmse,validation_folds
0,RB,47.208279,63.458947,5


### Walk-forward validation refactor

Converted the RB linear-regression evaluation into a reusable function.

Validation design:
- Position-specific models
- Training data limited to seasons before the validation season
- Validation seasons: 2021–2025
- Median imputation performed inside the modeling pipeline

Regression check:
- Previous RB MAE: 47.21
- Refactored RB MAE: 47.21
- Previous RB RMSE: 63.46
- Refactored RB RMSE: 63.46

Next step: define the appropriate feature lists and run the same framework
for QB, WR, and TE.

# Run it for each position

There's meaningful improvement from baseline to regression so now time to apply same method to all positions

In [139]:
base_features = [
    "age",
    "age_squared",
    "years_exp",
    "draft_number_filled",
    "undrafted_flag",
    "team_change_flag",

    "fantasy_points_lag_1",
    "fantasy_points_lag_2",
    "fantasy_ppg_lag_1",
    "fantasy_points_2yr_weighted",

    "games_lag_1",
    "games_2yr_avg",

    "fantasy_points_change"
]

In [140]:
qb_features = base_features + [
    "carries_lag_1",
    "carries_per_game_lag_1",
    "carries_2yr_avg"
]

In [141]:
rb_features = base_features + [
    "targets_lag_1",
    "carries_lag_1",
    "receptions_lag_1",
    "opportunities_lag_1",
    "targets_per_game_lag_1",
    "carries_per_game_lag_1",
    "opportunities_per_game_lag_1",
    "yards_per_carry_lag_1",
    "fantasy_points_per_opportunity_lag_1",
    "targets_2yr_avg",
    "carries_2yr_avg",
    "opportunities_2yr_avg"
]

In [142]:
wr_features = base_features + [
    "targets_lag_1",
    "receptions_lag_1",
    "targets_per_game_lag_1",
    "target_share_lag_1",
    "air_yards_share_lag_1",
    "wopr_lag_1",
    "targets_change",
    "yards_per_target_lag_1",
    "catch_rate_lag_1",
    "fantasy_points_per_opportunity_lag_1",
    "targets_2yr_avg",
    "target_share_2yr_avg",
    "wopr_2yr_avg"
]

In [143]:
te_features = wr_features.copy()

In [144]:
position_features = {
    "QB": qb_features,
    "RB": rb_features,
    "WR": wr_features,
    "TE": te_features
}

In [145]:
for position, features in position_features.items():
    missing = [feature for feature in features if feature not in model_data.columns]
    print(position, "missing features:", missing)

QB missing features: []
RB missing features: []
WR missing features: []
TE missing features: []


In [146]:
model_data.groupby(["position", "season"]).size().unstack(fill_value=0)

season,2018,2019,2020,2021,2022,2023,2024,2025
position,,,,,,,,
QB,41,44,54,54,62,61,56,58
RB,75,80,95,99,100,93,89,95
TE,68,75,81,80,78,80,79,94
WR,123,118,132,156,154,140,144,151


In [147]:
def evaluate_linear_model(data, features, target, validation_seasons):
    results = []

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LinearRegression())
    ])

    for validation_season in validation_seasons:

        train = data[data["season"] < validation_season]
        valid = data[data["season"] == validation_season]

        X_train = train[features]
        y_train = train[target]

        X_valid = valid[features]
        y_valid = valid[target]

        model.fit(X_train, y_train)

        predictions = model.predict(X_valid)

        mae = mean_absolute_error(y_valid, predictions)
        rmse = mean_squared_error(y_valid, predictions) ** 0.5

        baseline_predictions = valid["fantasy_points_2yr_weighted"]

        baseline_mae = mean_absolute_error(
            y_valid,
            baseline_predictions
        )

        baseline_rmse = mean_squared_error(
            y_valid,
            baseline_predictions
        ) ** 0.5

        results.append({
            "season": validation_season,
            "mae": mae,
            "rmse": rmse,
            "baseline_mae": baseline_mae,
            "baseline_rmse": baseline_rmse
        })

    return pd.DataFrame(results)

In [148]:
qb_df = model_data[model_data["position"] == "QB"].copy()
rb_df = model_data[model_data["position"] == "RB"].copy()
wr_df = model_data[model_data["position"] == "WR"].copy()
te_df = model_data[model_data["position"] == "TE"].copy()

In [149]:
qb_results = evaluate_linear_model(
    qb_df, qb_features, target, validation_seasons
)

rb_results = evaluate_linear_model(
    rb_df, rb_features, target, validation_seasons
)

wr_results = evaluate_linear_model(
    wr_df, wr_features, target, validation_seasons
)

te_results = evaluate_linear_model(
    te_df, te_features, target, validation_seasons
)

In [150]:
position_summary = pd.DataFrame({
    "QB": qb_results[
        ["mae", "rmse", "baseline_mae", "baseline_rmse"]
    ].mean(),

    "RB": rb_results[
        ["mae", "rmse", "baseline_mae", "baseline_rmse"]
    ].mean(),

    "WR": wr_results[
        ["mae", "rmse", "baseline_mae", "baseline_rmse"]
    ].mean(),

    "TE": te_results[
        ["mae", "rmse", "baseline_mae", "baseline_rmse"]
    ].mean()
}).T

position_summary

,mae,rmse,baseline_mae,baseline_rmse
QB,60.643768,80.208145,60.649041,84.162757
RB,47.208279,63.458947,49.277856,68.718076
WR,41.369868,54.354387,44.353908,59.434319
TE,29.545793,39.037937,29.496247,39.824319


In [156]:
qb_features_added = base_features + [
    # Rushing
    "carries_lag_1",
    "carries_per_game_lag_1",
    "carries_2yr_avg",

    # Passing
    "attempts_lag_1",
    "passing_yards_lag_1",
    "passing_tds_lag_1",
    "passing_interceptions_lag_1",
    "attempts_per_game_lag_1",
    "attempts_2yr_avg",
    "passing_yards_2yr_avg"
]

In [157]:
qb_df = model_data[
    model_data["position"] == "QB"
].copy()

In [158]:
qb_results = evaluate_linear_model(
    qb_df,
    qb_features_added,
    target,
    validation_seasons
)

In [159]:
qb_results[
    ["mae", "rmse", "baseline_mae", "baseline_rmse"]
].mean()

mae              61.897407
rmse             81.697873
baseline_mae     60.649041
baseline_rmse    84.162757
dtype: float64

Adding explicit QB passing-volume features did not improve linear-regression performance. MAE increased from 60.64 to 61.90 and RMSE increased from 80.21 to 81.70. The simpler feature set was retained for the linear baseline, while the additional passing features remain available for testing with regularized and nonlinear models.

In [155]:
qb_features = base_features + [
    "carries_lag_1",
    "carries_per_game_lag_1",
    "carries_2yr_avg"
]

## Ridge Regression

In [166]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

In [167]:
def evaluate_ridge_model(
    data,
    features,
    target,
    validation_seasons,
    alpha=1.0
):
    results = []

    model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=alpha))
    ])

    for validation_season in validation_seasons:

        train = data[data["season"] < validation_season]
        valid = data[data["season"] == validation_season]

        X_train = train[features]
        y_train = train[target]

        X_valid = valid[features]
        y_valid = valid[target]

        model.fit(X_train, y_train)

        predictions = model.predict(X_valid)

        mae = mean_absolute_error(y_valid, predictions)
        rmse = mean_squared_error(y_valid, predictions) ** 0.5

        baseline_predictions = valid["fantasy_points_2yr_weighted"]

        baseline_mae = mean_absolute_error(
            y_valid,
            baseline_predictions
        )

        baseline_rmse = mean_squared_error(
            y_valid,
            baseline_predictions
        ) ** 0.5

        results.append({
            "season": validation_season,
            "mae": mae,
            "rmse": rmse,
            "baseline_mae": baseline_mae,
            "baseline_rmse": baseline_rmse
        })

    return pd.DataFrame(results)

In [168]:
qb_ridge_results = evaluate_ridge_model(
    qb_df,
    qb_features,
    target,
    validation_seasons,
    alpha=1.0
)

rb_ridge_results = evaluate_ridge_model(
    rb_df,
    rb_features,
    target,
    validation_seasons,
    alpha=1.0
)

wr_ridge_results = evaluate_ridge_model(
    wr_df,
    wr_features,
    target,
    validation_seasons,
    alpha=1.0
)

te_ridge_results = evaluate_ridge_model(
    te_df,
    te_features,
    target,
    validation_seasons,
    alpha=1.0
)

In [169]:
ridge_summary = pd.DataFrame({
    "QB": qb_ridge_results[["mae", "rmse"]].mean(),
    "RB": rb_ridge_results[["mae", "rmse"]].mean(),
    "WR": wr_ridge_results[["mae", "rmse"]].mean(),
    "TE": te_ridge_results[["mae", "rmse"]].mean()
}).T

ridge_summary

,mae,rmse
QB,60.246708,79.613970
RB,46.698805,62.927449
WR,41.205141,54.160234
TE,29.488181,38.830578


In [164]:
position_summary[["mae", "rmse"]]

,mae,rmse
QB,60.643768,80.208145
RB,47.208279,63.458947
WR,41.369868,54.354387
TE,29.545793,39.037937


In [170]:
tuning_seasons = [2021, 2022, 2023, 2024]
test_season = 2025

In [177]:
alphas = [10, 30, 100, 300, 1000]

In [178]:
def tune_ridge_alpha(
    data,
    features,
    target,
    validation_seasons,
    alphas
):
    tuning_results = []

    for alpha in alphas:

        results = evaluate_ridge_model(
            data,
            features,
            target,
            validation_seasons,
            alpha=alpha
        )

        tuning_results.append({
            "alpha": alpha,
            "mean_mae": results["mae"].mean(),
            "mean_rmse": results["rmse"].mean()
        })

    return pd.DataFrame(tuning_results)

In [179]:
rb_ridge_tuning = tune_ridge_alpha(
    rb_df,
    rb_features,
    target,
    tuning_seasons,
    alphas
)

rb_ridge_tuning

,alpha,mean_mae,mean_rmse
0,10,45.613002,61.272279
1,30,45.268700,60.902014
2,100,45.317744,60.734502
3,300,45.892287,61.057217
4,1000,48.518169,62.758012


In [180]:
best_rb_alpha = 30

In [181]:
rb_2025_results = evaluate_ridge_model(
    rb_df,
    rb_features,
    target,
    [2025],
    alpha=best_rb_alpha
)

rb_2025_results

,season,mae,rmse,baseline_mae,baseline_rmse
0,2025,48.604497,67.269637,48.621789,68.609732


The tuned RB Ridge model selected on 2021–2024 validation seasons generalized to the 2025 holdout, slightly improving MAE (48.60 vs. 48.62) and more clearly improving RMSE (67.27 vs. 68.61) relative to the weighted historical baseline. This suggests modest but real value from regularization.

In [182]:
alphas = [0.01, 0.1, 1, 10, 30, 100, 300, 1000]

In [183]:
tuning_seasons = [2021, 2022, 2023, 2024]

# QB alpha Tuning

In [184]:
qb_ridge_tuning = tune_ridge_alpha(
    qb_df,
    qb_features,
    target,
    tuning_seasons,
    alphas
)

qb_ridge_tuning

,alpha,mean_mae,mean_rmse
0,0.01,61.458643,81.103929
1,0.10,61.398522,81.016912
2,1.00,60.958455,80.403120
3,10.00,60.577327,79.083036
4,30.00,61.046184,78.836513
5,100.00,61.312834,78.850022
6,300.00,62.930325,80.021665
7,1000.00,73.006714,86.890363


In [185]:
best_qb_alpha = 10

In [186]:
qb_2025_results = evaluate_ridge_model(
    qb_df,
    qb_features,
    target,
    [2025],
    alpha=best_qb_alpha
)

qb_2025_results

,season,mae,rmse,baseline_mae,baseline_rmse
0,2025,57.895089,76.303331,61.24569,83.36875


# WR alpha Tuning

In [187]:
wr_ridge_tuning = tune_ridge_alpha(
    wr_df,
    wr_features,
    target,
    tuning_seasons,
    alphas
)

wr_ridge_tuning

,alpha,mean_mae,mean_rmse
0,0.01,42.821510,55.597233
1,0.10,42.783578,55.559449
2,1.00,42.597491,55.339314
3,10.00,42.269596,54.856583
4,30.00,42.288950,54.775548
5,100.00,42.440691,54.825084
6,300.00,42.506811,54.920929
7,1000.00,43.362009,55.733005


In [189]:
best_wr_alpha = 10

In [190]:
wr_2025_results = evaluate_ridge_model(
    wr_df,
    wr_features,
    target,
    [2025],
    alpha=best_wr_alpha
)

wr_2025_results

,season,mae,rmse,baseline_mae,baseline_rmse
0,2025,35.9089,49.514784,41.007219,58.281402


# TE alpha Tuning

In [191]:
te_ridge_tuning = tune_ridge_alpha(
    te_df,
    te_features,
    target,
    tuning_seasons,
    alphas
)

te_ridge_tuning

,alpha,mean_mae,mean_rmse
0,0.01,30.110183,39.415634
1,0.10,30.127560,39.376072
2,1.00,30.071917,39.222115
3,10.00,29.636050,38.679744
4,30.00,29.638836,38.584602
5,100.00,29.929690,38.947496
6,300.00,30.602132,39.567314
7,1000.00,32.401179,41.277511


In [192]:
best_te_alpha = 10

In [193]:
te_2025_results = evaluate_ridge_model(
    te_df,
    te_features,
    target,
    [2025],
    alpha=best_te_alpha
)

te_2025_results

,season,mae,rmse,baseline_mae,baseline_rmse
0,2025,26.747866,36.725585,29.098979,39.488839


In [194]:
best_alphas = {
    "QB": 10,
    "RB": 30,
    "WR": 10,
    "TE": 10
}

# Ridge Holdout Summary

In [195]:
ridge_holdout_summary = pd.DataFrame({
    "QB": qb_2025_results.iloc[0],
    "RB": rb_2025_results.iloc[0],
    "WR": wr_2025_results.iloc[0],
    "TE": te_2025_results.iloc[0]
}).T

ridge_holdout_summary["mae_improvement"] = (
    ridge_holdout_summary["baseline_mae"]
    - ridge_holdout_summary["mae"]
)

ridge_holdout_summary["rmse_improvement"] = (
    ridge_holdout_summary["baseline_rmse"]
    - ridge_holdout_summary["rmse"]
)

ridge_holdout_summary[
    [
        "mae",
        "baseline_mae",
        "mae_improvement",
        "rmse",
        "baseline_rmse",
        "rmse_improvement"
    ]
]

,mae,baseline_mae,mae_improvement,rmse,baseline_rmse,rmse_improvement
QB,57.895089,61.245690,3.350601,76.303331,83.368750,7.065419
RB,48.604497,48.621789,0.017293,67.269637,68.609732,1.340095
WR,35.908900,41.007219,5.098319,49.514784,58.281402,8.766618
TE,26.747866,29.098979,2.351113,36.725585,39.488839,2.763254


Ridge Regression Conclusion: Position-specific Ridge models were tuned using 2021–2024 walk-forward validation and evaluated on an untouched 2025 holdout season. Ridge outperformed the weighted historical-production baseline at all four positions on RMSE and at all four positions on MAE, although the RB MAE improvement was negligible. The strongest gains occurred for WR and QB.

# Random Forest

In [196]:
from sklearn.ensemble import RandomForestRegressor

In [197]:
def evaluate_random_forest(
    data,
    features,
    target,
    validation_seasons,
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    max_features="sqrt"
):
    results = []

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=42,
            n_jobs=-1
        ))
    ])

    for validation_season in validation_seasons:

        train = data[data["season"] < validation_season]
        valid = data[data["season"] == validation_season]

        X_train = train[features]
        y_train = train[target]

        X_valid = valid[features]
        y_valid = valid[target]

        model.fit(X_train, y_train)

        predictions = model.predict(X_valid)

        mae = mean_absolute_error(y_valid, predictions)
        rmse = mean_squared_error(y_valid, predictions) ** 0.5

        results.append({
            "season": validation_season,
            "mae": mae,
            "rmse": rmse
        })

    return pd.DataFrame(results)

In [198]:
rb_rf_results = evaluate_random_forest(
    rb_df,
    rb_features,
    target,
    tuning_seasons
)

rb_rf_results

,season,mae,rmse
0,2021,46.560077,59.767063
1,2022,48.416590,65.126783
2,2023,42.831837,56.168919
3,2024,45.250893,61.139740


In [199]:
rb_rf_results[["mae", "rmse"]].mean()

mae     45.764849
rmse    60.550626
dtype: float64

### Tuning RF Hyperparameters

In [200]:
rf_configs = [
    {"max_depth": None, "min_samples_leaf": 1},
    {"max_depth": None, "min_samples_leaf": 3},
    {"max_depth": None, "min_samples_leaf": 5},
    {"max_depth": 10, "min_samples_leaf": 1},
    {"max_depth": 10, "min_samples_leaf": 3},
    {"max_depth": 10, "min_samples_leaf": 5},
    {"max_depth": 6, "min_samples_leaf": 3},
    {"max_depth": 6, "min_samples_leaf": 5},
]

In [201]:
rf_tuning_results = []

for config in rf_configs:

    results = evaluate_random_forest(
        rb_df,
        rb_features,
        target,
        tuning_seasons,
        n_estimators=300,
        max_depth=config["max_depth"],
        min_samples_leaf=config["min_samples_leaf"],
        max_features="sqrt"
    )

    rf_tuning_results.append({
        "max_depth": config["max_depth"],
        "min_samples_leaf": config["min_samples_leaf"],
        "mean_mae": results["mae"].mean(),
        "mean_rmse": results["rmse"].mean()
    })

rb_rf_tuning = pd.DataFrame(rf_tuning_results)

rb_rf_tuning.sort_values("mean_mae")

,max_depth,min_samples_leaf,mean_mae,mean_rmse
5,10.0,5,45.025360,60.101112
2,NaN,5,45.058527,60.142987
7,6.0,5,45.100283,60.166687
1,NaN,3,45.336788,60.476238
6,6.0,3,45.345296,60.397228
4,10.0,3,45.397972,60.448893
0,NaN,1,45.764849,60.550626
3,10.0,1,45.792668,60.751799


In [202]:
rb_rf_2025 = evaluate_random_forest(
    rb_df,
    rb_features,
    target,
    [2025],
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    max_features="sqrt"
)

rb_rf_2025

,season,mae,rmse
0,2025,49.166318,68.50537


# QB Hyperparameter Tuning

In [205]:
rf_tuning_results = []

for config in rf_configs:

    results = evaluate_random_forest(
        qb_df,
        qb_features,
        target,
        tuning_seasons,
        n_estimators=300,
        max_depth=config["max_depth"],
        min_samples_leaf=config["min_samples_leaf"],
        max_features="sqrt"
    )

    rf_tuning_results.append({
        "max_depth": config["max_depth"],
        "min_samples_leaf": config["min_samples_leaf"],
        "mean_mae": results["mae"].mean(),
        "mean_rmse": results["rmse"].mean()
    })

qb_rf_tuning = pd.DataFrame(rf_tuning_results)

qb_rf_tuning.sort_values("mean_mae")

,max_depth,min_samples_leaf,mean_mae,mean_rmse
7,6.0,5,57.581973,78.707772
6,6.0,3,57.621683,78.633301
1,NaN,3,57.646817,78.829955
2,NaN,5,57.681210,79.036356
5,10.0,5,57.681904,79.027177
4,10.0,3,57.723567,78.780399
0,NaN,1,58.114693,79.119863
3,10.0,1,58.192144,78.836238


In [204]:
qb_rf_2025 = evaluate_random_forest(
    qb_df,
    qb_features,
    target,
    [2025],
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=5,
    max_features="sqrt"
)

qb_rf_2025

,season,mae,rmse
0,2025,60.577766,78.424328


# WR Hyperparameter Tuning

In [206]:
rf_tuning_results = []

for config in rf_configs:

    results = evaluate_random_forest(
        wr_df,
        wr_features,
        target,
        tuning_seasons,
        n_estimators=300,
        max_depth=config["max_depth"],
        min_samples_leaf=config["min_samples_leaf"],
        max_features="sqrt"
    )

    rf_tuning_results.append({
        "max_depth": config["max_depth"],
        "min_samples_leaf": config["min_samples_leaf"],
        "mean_mae": results["mae"].mean(),
        "mean_rmse": results["rmse"].mean()
    })

wr_rf_tuning = pd.DataFrame(rf_tuning_results)

wr_rf_tuning.sort_values("mean_mae")

,max_depth,min_samples_leaf,mean_mae,mean_rmse
4,10.0,3,41.311828,54.973413
3,10.0,1,41.354957,55.108703
6,6.0,3,41.382125,54.843097
1,NaN,3,41.430646,55.019256
7,6.0,5,41.482053,54.903321
2,NaN,5,41.541566,55.085623
5,10.0,5,41.565986,55.115128
0,NaN,1,41.701635,55.472536


In [207]:
wr_rf_2025 = evaluate_random_forest(
    wr_df,
    wr_features,
    target,
    [2025],
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=3,
    max_features="sqrt"
)

wr_rf_2025

,season,mae,rmse
0,2025,36.782335,49.566862


# TE Hyperparameter Tuning

In [208]:
rf_tuning_results = []

for config in rf_configs:

    results = evaluate_random_forest(
        te_df,
        te_features,
        target,
        tuning_seasons,
        n_estimators=300,
        max_depth=config["max_depth"],
        min_samples_leaf=config["min_samples_leaf"],
        max_features="sqrt"
    )

    rf_tuning_results.append({
        "max_depth": config["max_depth"],
        "min_samples_leaf": config["min_samples_leaf"],
        "mean_mae": results["mae"].mean(),
        "mean_rmse": results["rmse"].mean()
    })

te_rf_tuning = pd.DataFrame(rf_tuning_results)

te_rf_tuning.sort_values("mean_mae")

,max_depth,min_samples_leaf,mean_mae,mean_rmse
5,10.0,5,30.788940,39.257894
7,6.0,5,30.801604,39.360404
2,NaN,5,30.822201,39.284887
6,6.0,3,30.966513,39.557429
1,NaN,3,31.101040,39.516140
4,10.0,3,31.107640,39.417147
3,10.0,1,31.411780,39.953436
0,NaN,1,31.446599,39.883009


In [209]:
te_rf_2025 = evaluate_random_forest(
    te_df,
    te_features,
    target,
    [2025],
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    max_features="sqrt"
)

te_rf_2025

,season,mae,rmse
0,2025,25.78808,35.119637


# Model Comparisons So Far

In [210]:
model_holdout_comparison = pd.DataFrame({
    "position": ["QB", "RB", "WR", "TE"],

    "ridge_mae": [
        qb_2025_results["mae"].iloc[0],
        rb_2025_results["mae"].iloc[0],
        wr_2025_results["mae"].iloc[0],
        te_2025_results["mae"].iloc[0]
    ],

    "rf_mae": [
        qb_rf_2025["mae"].iloc[0],
        rb_rf_2025["mae"].iloc[0],
        wr_rf_2025["mae"].iloc[0],
        te_rf_2025["mae"].iloc[0]
    ],

    "ridge_rmse": [
        qb_2025_results["rmse"].iloc[0],
        rb_2025_results["rmse"].iloc[0],
        wr_2025_results["rmse"].iloc[0],
        te_2025_results["rmse"].iloc[0]
    ],

    "rf_rmse": [
        qb_rf_2025["rmse"].iloc[0],
        rb_rf_2025["rmse"].iloc[0],
        wr_rf_2025["rmse"].iloc[0],
        te_rf_2025["rmse"].iloc[0]
    ]
})

model_holdout_comparison

,position,ridge_mae,rf_mae,ridge_rmse,rf_rmse
0,QB,57.895089,60.577766,76.303331,78.424328
1,RB,48.604497,49.166318,67.269637,68.505370
2,WR,35.908900,36.782335,49.514784,49.566862
3,TE,26.747866,25.788080,36.725585,35.119637


Random Forest conclusion: Random Forest did not consistently outperform Ridge on the 2025 holdout. Ridge remained stronger for QB, RB, and WR, while Random Forest produced the best TE performance. This suggests that model performance should be evaluated and selected separately by position rather than assuming one algorithm is universally best.

# Gradient Boosting

In [211]:
from sklearn.ensemble import GradientBoostingRegressor

In [212]:
def evaluate_gradient_boosting(
    data,
    features,
    target,
    validation_seasons,
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    min_samples_leaf=3
):
    results = []

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingRegressor(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        ))
    ])

    for validation_season in validation_seasons:

        train = data[data["season"] < validation_season]
        valid = data[data["season"] == validation_season]

        X_train = train[features]
        y_train = train[target]

        X_valid = valid[features]
        y_valid = valid[target]

        model.fit(X_train, y_train)

        predictions = model.predict(X_valid)

        mae = mean_absolute_error(y_valid, predictions)
        rmse = mean_squared_error(y_valid, predictions) ** 0.5

        results.append({
            "season": validation_season,
            "mae": mae,
            "rmse": rmse
        })

    return pd.DataFrame(results)

In [213]:
rb_gb_results = evaluate_gradient_boosting(
    rb_df,
    rb_features,
    target,
    tuning_seasons
)

rb_gb_results

,season,mae,rmse
0,2021,51.546757,65.437975
1,2022,47.709748,63.429311
2,2023,39.305673,53.990787
3,2024,45.434237,63.588200


In [214]:
rb_gb_results[["mae", "rmse"]].mean()

mae     45.999104
rmse    61.611568
dtype: float64

In [215]:
gb_configs = [
    {
        "n_estimators": 100,
        "learning_rate": 0.05,
        "max_depth": 2,
        "min_samples_leaf": 3
    },
    {
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": 2,
        "min_samples_leaf": 3
    },
    {
        "n_estimators": 300,
        "learning_rate": 0.03,
        "max_depth": 2,
        "min_samples_leaf": 3
    },
    {
        "n_estimators": 100,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_samples_leaf": 5
    },
    {
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_samples_leaf": 5
    },
    {
        "n_estimators": 300,
        "learning_rate": 0.03,
        "max_depth": 3,
        "min_samples_leaf": 5
    }
]

In [216]:
gb_tuning_results = []

for config in gb_configs:

    results = evaluate_gradient_boosting(
        rb_df,
        rb_features,
        target,
        tuning_seasons,
        n_estimators=config["n_estimators"],
        learning_rate=config["learning_rate"],
        max_depth=config["max_depth"],
        min_samples_leaf=config["min_samples_leaf"]
    )

    gb_tuning_results.append({
        "n_estimators": config["n_estimators"],
        "learning_rate": config["learning_rate"],
        "max_depth": config["max_depth"],
        "min_samples_leaf": config["min_samples_leaf"],
        "mean_mae": results["mae"].mean(),
        "mean_rmse": results["rmse"].mean()
    })

rb_gb_tuning = pd.DataFrame(gb_tuning_results)

rb_gb_tuning.sort_values("mean_mae")

,n_estimators,learning_rate,max_depth,min_samples_leaf,mean_mae,mean_rmse
1,200,0.05,2,3,45.492330,60.640328
3,100,0.05,3,5,45.638755,60.166572
2,300,0.03,2,3,45.734030,60.653429
0,100,0.05,2,3,45.924665,60.327348
5,300,0.03,3,5,45.982439,61.113352
4,200,0.05,3,5,46.077644,61.265189


In [217]:
rb_gb_2025 = evaluate_gradient_boosting(
    rb_df,
    rb_features,
    target,
    [2025],
    n_estimators=200,
    learning_rate=0.05,
    max_depth=2,
    min_samples_leaf=3
)

rb_gb_2025

,season,mae,rmse
0,2025,51.277702,72.552754


The tuned Gradient Boosting model did not generalize well to the 2025 RB holdout, producing higher MAE and RMSE than Ridge, Random Forest, and the weighted historical baseline. Ridge remains the strongest RB model on the holdout.

# QB Gradient Boosting Tuning

In [218]:
gb_tuning_results = []

for config in gb_configs:

    results = evaluate_gradient_boosting(
        qb_df,
        qb_features,
        target,
        tuning_seasons,
        n_estimators=config["n_estimators"],
        learning_rate=config["learning_rate"],
        max_depth=config["max_depth"],
        min_samples_leaf=config["min_samples_leaf"]
    )

    gb_tuning_results.append({
        "n_estimators": config["n_estimators"],
        "learning_rate": config["learning_rate"],
        "max_depth": config["max_depth"],
        "min_samples_leaf": config["min_samples_leaf"],
        "mean_mae": results["mae"].mean(),
        "mean_rmse": results["rmse"].mean()
    })

qb_gb_tuning = pd.DataFrame(gb_tuning_results)

qb_gb_tuning.sort_values("mean_mae")

,n_estimators,learning_rate,max_depth,min_samples_leaf,mean_mae,mean_rmse
2,300,0.03,2,3,57.309172,78.554914
1,200,0.05,2,3,57.666221,79.100116
0,100,0.05,2,3,57.749268,78.512680
5,300,0.03,3,5,57.928250,79.005874
4,200,0.05,3,5,57.961436,79.792854
3,100,0.05,3,5,58.775832,79.695972


In [219]:
qb_gb_2025 = evaluate_gradient_boosting(
    qb_df,
    qb_features,
    target,
    [2025],
    n_estimators=300,
    learning_rate=0.03,
    max_depth=2,
    min_samples_leaf=3
)

qb_gb_2025

,season,mae,rmse
0,2025,62.854153,81.960783


# WR Gradient Boost Tuning

In [220]:
gb_tuning_results = []

for config in gb_configs:

    results = evaluate_gradient_boosting(
        wr_df,
        wr_features,
        target,
        tuning_seasons,
        n_estimators=config["n_estimators"],
        learning_rate=config["learning_rate"],
        max_depth=config["max_depth"],
        min_samples_leaf=config["min_samples_leaf"]
    )

    gb_tuning_results.append({
        "n_estimators": config["n_estimators"],
        "learning_rate": config["learning_rate"],
        "max_depth": config["max_depth"],
        "min_samples_leaf": config["min_samples_leaf"],
        "mean_mae": results["mae"].mean(),
        "mean_rmse": results["rmse"].mean()
    })

wr_gb_tuning = pd.DataFrame(gb_tuning_results)

wr_gb_tuning.sort_values("mean_mae")

,n_estimators,learning_rate,max_depth,min_samples_leaf,mean_mae,mean_rmse
2,300,0.03,2,3,42.424405,56.755267
0,100,0.05,2,3,42.477608,55.839996
1,200,0.05,2,3,42.636162,57.247698
3,100,0.05,3,5,43.024042,57.098330
5,300,0.03,3,5,43.172677,57.936545
4,200,0.05,3,5,43.190081,58.310444


In [221]:
wr_gb_2025 = evaluate_gradient_boosting(
    wr_df,
    wr_features,
    target,
    [2025],
    n_estimators=300,
    learning_rate=0.03,
    max_depth=2,
    min_samples_leaf=3
)

wr_gb_2025

,season,mae,rmse
0,2025,36.193417,49.877893


# TE Gradient Boosting Tuning

In [222]:
gb_tuning_results = []

for config in gb_configs:

    results = evaluate_gradient_boosting(
        te_df,
        te_features,
        target,
        tuning_seasons,
        n_estimators=config["n_estimators"],
        learning_rate=config["learning_rate"],
        max_depth=config["max_depth"],
        min_samples_leaf=config["min_samples_leaf"]
    )

    gb_tuning_results.append({
        "n_estimators": config["n_estimators"],
        "learning_rate": config["learning_rate"],
        "max_depth": config["max_depth"],
        "min_samples_leaf": config["min_samples_leaf"],
        "mean_mae": results["mae"].mean(),
        "mean_rmse": results["rmse"].mean()
    })

te_gb_tuning = pd.DataFrame(gb_tuning_results)

te_gb_tuning.sort_values("mean_mae")

,n_estimators,learning_rate,max_depth,min_samples_leaf,mean_mae,mean_rmse
3,100,0.05,3,5,31.131345,39.325175
0,100,0.05,2,3,31.312416,39.816298
5,300,0.03,3,5,31.373222,40.055485
4,200,0.05,3,5,31.554123,40.256755
2,300,0.03,2,3,31.572869,40.144492
1,200,0.05,2,3,31.594429,40.400616


In [223]:
te_gb_2025 = evaluate_gradient_boosting(
    te_df,
    te_features,
    target,
    [2025],
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    min_samples_leaf=5
)

te_gb_2025

,season,mae,rmse
0,2025,26.971655,37.96348


# Final Model Comparison

In [224]:
qb_linear_2025 = qb_results[qb_results["season"] == 2025]
rb_linear_2025 = rb_results[rb_results["season"] == 2025]
wr_linear_2025 = wr_results[wr_results["season"] == 2025]
te_linear_2025 = te_results[te_results["season"] == 2025]

In [225]:
final_model_comparison = pd.DataFrame([
    # QB
    {
        "position": "QB",
        "model": "Linear Regression",
        "mae": qb_linear_2025["mae"].iloc[0],
        "rmse": qb_linear_2025["rmse"].iloc[0]
    },
    {
        "position": "QB",
        "model": "Ridge",
        "mae": qb_2025_results["mae"].iloc[0],
        "rmse": qb_2025_results["rmse"].iloc[0]
    },
    {
        "position": "QB",
        "model": "Random Forest",
        "mae": qb_rf_2025["mae"].iloc[0],
        "rmse": qb_rf_2025["rmse"].iloc[0]
    },
    {
        "position": "QB",
        "model": "Gradient Boosting",
        "mae": qb_gb_2025["mae"].iloc[0],
        "rmse": qb_gb_2025["rmse"].iloc[0]
    },

    # RB
    {
        "position": "RB",
        "model": "Linear Regression",
        "mae": rb_linear_2025["mae"].iloc[0],
        "rmse": rb_linear_2025["rmse"].iloc[0]
    },
    {
        "position": "RB",
        "model": "Ridge",
        "mae": rb_2025_results["mae"].iloc[0],
        "rmse": rb_2025_results["rmse"].iloc[0]
    },
    {
        "position": "RB",
        "model": "Random Forest",
        "mae": rb_rf_2025["mae"].iloc[0],
        "rmse": rb_rf_2025["rmse"].iloc[0]
    },
    {
        "position": "RB",
        "model": "Gradient Boosting",
        "mae": rb_gb_2025["mae"].iloc[0],
        "rmse": rb_gb_2025["rmse"].iloc[0]
    },

    # WR
    {
        "position": "WR",
        "model": "Linear Regression",
        "mae": wr_linear_2025["mae"].iloc[0],
        "rmse": wr_linear_2025["rmse"].iloc[0]
    },
    {
        "position": "WR",
        "model": "Ridge",
        "mae": wr_2025_results["mae"].iloc[0],
        "rmse": wr_2025_results["rmse"].iloc[0]
    },
    {
        "position": "WR",
        "model": "Random Forest",
        "mae": wr_rf_2025["mae"].iloc[0],
        "rmse": wr_rf_2025["rmse"].iloc[0]
    },
    {
        "position": "WR",
        "model": "Gradient Boosting",
        "mae": wr_gb_2025["mae"].iloc[0],
        "rmse": wr_gb_2025["rmse"].iloc[0]
    },

    # TE
    {
        "position": "TE",
        "model": "Linear Regression",
        "mae": te_linear_2025["mae"].iloc[0],
        "rmse": te_linear_2025["rmse"].iloc[0]
    },
    {
        "position": "TE",
        "model": "Ridge",
        "mae": te_2025_results["mae"].iloc[0],
        "rmse": te_2025_results["rmse"].iloc[0]
    },
    {
        "position": "TE",
        "model": "Random Forest",
        "mae": te_rf_2025["mae"].iloc[0],
        "rmse": te_rf_2025["rmse"].iloc[0]
    },
    {
        "position": "TE",
        "model": "Gradient Boosting",
        "mae": te_gb_2025["mae"].iloc[0],
        "rmse": te_gb_2025["rmse"].iloc[0]
    }
])

final_model_comparison.sort_values(
    ["position", "mae"]
)

,position,model,mae,rmse
1,QB,Ridge,57.895089,76.303331
0,QB,Linear Regression,58.178648,78.638270
2,QB,Random Forest,60.577766,78.424328
3,QB,Gradient Boosting,62.854153,81.960783
5,RB,Ridge,48.604497,67.269637
4,RB,Linear Regression,48.735904,66.178768
6,RB,Random Forest,49.166318,68.505370
7,RB,Gradient Boosting,51.277702,72.552754
14,TE,Random Forest,25.788080,35.119637
13,TE,Ridge,26.747866,36.725585


In [226]:
selected_models = {
    "QB": "Ridge",
    "RB": "Ridge",
    "WR": "Ridge",
    "TE": "Random Forest"
}

In [227]:
final_model_comparison["selected"] = (
    final_model_comparison.apply(
        lambda row:
        row["model"] == selected_models[row["position"]],
        axis=1
    )
)

final_model_comparison.sort_values(
    ["position", "mae"]
)

,position,model,mae,rmse,selected
1,QB,Ridge,57.895089,76.303331,True
0,QB,Linear Regression,58.178648,78.638270,False
2,QB,Random Forest,60.577766,78.424328,False
3,QB,Gradient Boosting,62.854153,81.960783,False
5,RB,Ridge,48.604497,67.269637,True
4,RB,Linear Regression,48.735904,66.178768,False
6,RB,Random Forest,49.166318,68.505370,False
7,RB,Gradient Boosting,51.277702,72.552754,False
14,TE,Random Forest,25.788080,35.119637,True
13,TE,Ridge,26.747866,36.725585,False


In [228]:
final_model_comparison.to_csv(
    "../reports/final_model_comparison.csv",
    index=False
)

Final model selection: Model selection was based on performance on the untouched 2025 holdout season after hyperparameter tuning on 2021–2024 walk-forward validation. Ridge Regression was selected for QB, RB, and WR, while Random Forest was selected for TE.

# Training Final Models

In [229]:
from sklearn.base import clone

In [230]:
final_qb_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10))
])

final_rb_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=30))
])

final_wr_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10))
])

In [231]:
final_te_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=5,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1
    ))
])

In [232]:
final_qb_model.fit(
    qb_df[qb_features],
    qb_df[target]
)

final_rb_model.fit(
    rb_df[rb_features],
    rb_df[target]
)

final_wr_model.fit(
    wr_df[wr_features],
    wr_df[target]
)

final_te_model.fit(
    te_df[te_features],
    te_df[target]
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](26,)","['age','age_squared','years_exp',...,'targets_2yr_avg', 'target_share_2yr_avg','wopr_2yr_avg']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,26
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or 